# Ekstrakcija podataka o kalcijumskim odgovorima iz originalnih fajlova

U ovom notebook-u, biće izvršena ekstrakcija vrednosti iz .xlsx fajlova koji sadrže informacije o fluorescenciji, parametre kalcijumskog odgovora, kao i parametre fita tog odgovora, za prvi odgovor na kalijum i prvi odgovor na CSF.

In [1]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

In [2]:
# Definisanje strukture projekta
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
UNPACKED_DIR = PROJECT_ROOT / "data" / "unpacked" / "1_25"
EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"

ZIP_PATH = RAW_DIR / "1_25.zip"

UNPACKED_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Ekstrakcija .xlsx fajlova iz ZIP arhive
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(UNPACKED_DIR)

In [4]:
# Lista .xlsx fajlova
xlsx_files = sorted(UNPACKED_DIR.rglob("*.xlsx"))

for path in xlsx_files:
    print(path.relative_to(UNPACKED_DIR))

10a_1..25/2023_11_01_CGNs_isolated3010_CSF_3.xlsx
10b_1..25/2023_07_10_CGNs_isolated0807_CSF_2_480nm-fitc_svi_ROI_min0.1.xlsx
10b_1..25/2023_07_10_CGNs_isolated0807_CSF_9_480nm-fitc_svi_ROI_min0.1.xlsx
3_1..25/2024_02_16_CGNs_iso1302_4.xlsx
4_1..25/2023_11_11_CGNs_isolated0911_CSF_5_min0.1.xlsx
7_1..25/2023_07_17_CGNs_isolated1407_CSF_5_480nm-fitc_svi_ROI_min0.1.xlsx


Vidimo da su svi fajlovi uspešno ekstrahovani iz arhive.

## Probna ekstrakcija vrednosti iz jednog fajla

U svrhu uspešne ekstrakcije vrednosti iz svih fajlova, biće izvršena probna ekstrakcija iz eksperimenta sa CSF-om pacijenta br. 3, označenog kao **2024_02_16_CGNs_iso1302_4**.

In [5]:
# Definisanje puta do fajla i provera
example_path = (
    UNPACKED_DIR
    / "3_1..25"
    / "2024_02_16_CGNs_iso1302_4.xlsx"
)

print(example_path.exists())

True


In [6]:
def inspect_workbook(path: Path) -> pd.DataFrame:
    """
    Vraća pregled svakog lista u Excel fajlu.
    """
    excel_file = pd.ExcelFile(path)

    rows = []

    for sheet_name in excel_file.sheet_names:
        sheet = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=None,
        )

        rows.append(
            {
                "sheet_name": sheet_name,
                "n_rows": sheet.shape[0],
                "n_columns": sheet.shape[1],
                "first_cell": sheet.iat[0, 0],
            }
        )

    return pd.DataFrame(rows)

In [7]:
overview = inspect_workbook(example_path)
overview

,sheet_name,n_rows,n_columns,first_cell
0,Sheet1,1271,168,1
1,Sheet2,1271,168,0
2,Sheet3,47,168,prvi kalijum
3,Sheet4,167,17,free coef
4,Sheet5,47,168,drugi kalijum
5,Sheet6,167,17,free coef
6,Sheet7,47,168,prvi CSF
7,Sheet8,167,17,free coef


Vidimo pregled dimenzija svakog lista, zajedno sa prvom ćelijom svakog, koja daje informacije o onome što se u listu nalazi.

### Ispitivanje i ekstrakcija podataka o odgovoru na K<sup>+</sup>

#### Ekstrakcija parametara kalcijumskog odgovora

In [8]:
k_summary = pd.read_excel(
    example_path,
    sheet_name="Sheet3",
    header=None,
)

print(k_summary.shape)

(47, 168)


Prva kolona sadrži informacije o tretmanu (npr. prvi kalijum), kao i oznake vrednosti koje se nalaze u redovima ispod njih.

In [9]:
k_summary.iloc[:, 0]

0      prvi kalijum
1             meanA
2          4.640796
3               NaN
4               NaN
5               NaN
6               NaN
7               NaN
8               NaN
9               NaN
10              NaN
11              NaN
12     amplitude K1
13              NaN
14              NaN
15              NaN
16              NaN
17              NaN
18              NaN
19              NaN
20              NaN
21              NaN
22              NaN
23    polusirine K1
24              NaN
25              NaN
26              NaN
27              NaN
28              NaN
29              NaN
30              NaN
31              NaN
32              NaN
33              NaN
34      lokacije K1
35              NaN
36              NaN
37              NaN
38              NaN
39              NaN
40              NaN
41              NaN
42              NaN
43              NaN
44              NaN
45      integral K1
46              NaN
Name: 0, dtype: object

Može biti korisno da imamo način da prevedemo numeričku vrednost kolone u originalni oblik sa slovima, ako budu bile potrebne provere.

In [10]:
def excel_column_letter(column_number: int) -> str:
    """
    Prevodi kolonu Excel fajla u slovni oblik.

    Primer:
        1  -> A
        2  -> B
        26 -> Z
        27 -> AA
    """
    letters = ""

    while column_number:
        column_number, remainder = divmod(column_number - 1, 26)
        letters = chr(65 + remainder) + letters

    return letters

In [11]:
print(excel_column_letter(1))
print(excel_column_letter(2))
print(excel_column_letter(27))

A
B
AA


Sada možemo definisati opštu funkciju za izvlačenje parametara kalcijumskog odgovora sa datog Excel lista.

In [12]:
def extract_summary_sheet(
    path: Path,
    sheet_name: str,
    prefix: str,
) -> pd.DataFrame:
    """
    Izvlači parametre odgovora prvog pika, kao i broj pikova iz lista sa
    horizontalno prikazanim parametrima.

    Svaki ROI se skladišti u jednoj Excel koloni, počevši od kolone B.
    """
    sheet = pd.read_excel(
        path,
        sheet_name=sheet_name,
        header=None,
    )

    # Kolona A sadrži oznake, a podaci počinju od kolone B
    roi_columns = sheet.columns[1:]

    # Na ovom listu je odvojeno deset redova za moguće pikove. 
    # Prvi red u svakoj grupi sadrži prvi pik
    amplitude_block = sheet.loc[13:22, roi_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

    output = pd.DataFrame(
        {
            "roi_id": [
                f"ROI_{index:03d}"
                for index in range(1, len(roi_columns) + 1)
            ],
            "roi_source_excel_column": [
                excel_column_letter(column_index + 1)
                for column_index in roi_columns
            ],
            f"{prefix}_amplitude": pd.to_numeric(
                sheet.loc[13, roi_columns],
                errors="coerce",
            ).to_numpy(),
            f"{prefix}_half_width": pd.to_numeric(
                sheet.loc[24, roi_columns],
                errors="coerce",
            ).to_numpy(),
            f"{prefix}_peak_location": pd.to_numeric(
                sheet.loc[35, roi_columns],
                errors="coerce",
            ).to_numpy(),
            f"{prefix}_integral": pd.to_numeric(
                sheet.loc[46, roi_columns],
                errors="coerce",
            ).to_numpy(),
            f"{prefix}_peak_count": (
                amplitude_block.notna().sum(axis=0).to_numpy()
            ),
        }
    )

    output[f"{prefix}_multiple_responses"] = (
        output[f"{prefix}_peak_count"] > 1
    )

    output[f"{prefix}_response_detected"] = (
        output[f"{prefix}_amplitude"].notna()
    )

    return output

In [13]:
# Primena funkcije na prvi kalijumski odgovor
k_features = extract_summary_sheet(
    path=example_path,
    sheet_name="Sheet3",
    prefix="k_ref",
)

k_features.head()

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_half_width,k_ref_peak_location,k_ref_integral,k_ref_peak_count,k_ref_multiple_responses,k_ref_response_detected
0,ROI_001,B,4.132505,22.884204,204.0,145.734835,1,False,True
1,ROI_002,C,2.346397,17.929824,204.0,60.761753,1,False,True
2,ROI_003,D,6.263296,9.635272,203.0,136.567826,1,False,True
3,ROI_004,E,4.667348,12.775321,204.0,96.063984,1,False,True
4,ROI_005,F,4.415487,10.662049,204.0,77.233380,1,False,True


In [14]:
k_features.shape

(167, 9)

Dimenzije se poklapaju sa brojem vrednosti u originalnom Excel fajlu.

Sada ćemo definisati funkciju za proveru normalizovanih vrednosti fluorescencije koje se u svim fajlovima nalaze na drugom listu.

In [15]:
# Učitavanje lista sa normalizovanim vrednostima i provera dimenzija
normalized_sheet = pd.read_excel(
    example_path,
    sheet_name="Sheet2",
    header=None,
)

normalized_sheet.shape

(1271, 168)

In [16]:
def extract_trace_qc(path: Path) -> pd.DataFrame:
    """
    Računa udeo negativnih vrednosti u kolonama lista sa normalizovanim
    podacima o intenzitetu fluorescencije u ćelijama.

    Svaki ROI počinje u Excel koloni B.
    """
    sheet = pd.read_excel(
        path,
        sheet_name="Sheet2",
        header=None,
    )

    traces = sheet.iloc[:, 1:].apply(
        pd.to_numeric,
        errors="coerce",
    )

    negative_fraction = (
        traces.lt(0).sum(axis=0)
        / traces.notna().sum(axis=0)
    )

    output = pd.DataFrame(
        {
            "roi_id": [
                f"ROI_{index:03d}"
                for index in range(1, traces.shape[1] + 1)
            ],
            "roi_source_excel_column": [
                excel_column_letter(column_index + 1)
                for column_index in traces.columns
            ],
            "normalized_negative_fraction": negative_fraction.to_numpy(),
        }
    )

    output["majority_negative_trace"] = (
        output["normalized_negative_fraction"] > 0.5
    )

    return output

In [17]:
trace_qc = extract_trace_qc(example_path)

trace_qc.head()

,roi_id,roi_source_excel_column,normalized_negative_fraction,majority_negative_trace
0,ROI_001,B,0.077891,False
1,ROI_002,C,0.095201,False
2,ROI_003,D,0.013375,False
3,ROI_004,E,0.118017,False
4,ROI_005,F,0.060582,False


Vidimo da smo dobili tabelu koja daje udeo negativnih vrednosti za svaku ćeliju.

Sada ćemo spojiti prethodnu tabelu sa parametrima kalijumskog odgovora sa novom.

In [18]:
example_features = k_features.merge(
    trace_qc,
    on=[
        "roi_id",
        "roi_source_excel_column",
    ],
    how="outer",
    validate="one_to_one", # proverava da li se svaki ROI pojavljuje
                           # tačno po jednom u svakoj tabeli
)

example_features.head()

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_half_width,k_ref_peak_location,k_ref_integral,k_ref_peak_count,k_ref_multiple_responses,k_ref_response_detected,normalized_negative_fraction,majority_negative_trace
0,ROI_001,B,4.132505,22.884204,204.0,145.734835,1,False,True,0.077891,False
1,ROI_002,C,2.346397,17.929824,204.0,60.761753,1,False,True,0.095201,False
2,ROI_003,D,6.263296,9.635272,203.0,136.567826,1,False,True,0.013375,False
3,ROI_004,E,4.667348,12.775321,204.0,96.063984,1,False,True,0.118017,False
4,ROI_005,F,4.415487,10.662049,204.0,77.233380,1,False,True,0.060582,False


In [19]:
example_features.shape

(167, 11)

Dobili smo očekivane dimenzije tabele, i vidimo da izgleda kako treba.

Sada ćemo ispitati povezanost ROI-eva sa većinom negativnom fluorescencijom sa nedostajućim pikovima kalijumskog odgovora.

In [20]:
pd.crosstab(
    example_features["majority_negative_trace"],
    example_features["k_ref_amplitude"].isna(),
    rownames=["majority_negative_trace"],
    colnames=["k_ref_amplitude_missing"],
    margins=True,
)

k_ref_amplitude_missing,False,True,All
majority_negative_trace,,,
False,163,0,163
True,1,3,4
All,164,3,167


Ovde vidimo da postoji: 
- 163 ROI-a koji prema udelu negativnih vrednosti deluju validno i pokazuju odgovor
- 3 ROI-a sa većinom negativnim vrednostima koji ne pokazuju odgovor
- 1 ROI sa većinom negativnim vrednostima koji i dalje pokazuje odgovor

Zasad možemo primeniti pravilo o isključenju onih ROI-eva koji imaju većinom negativne vrednosti fluorescencije, i dodati kolone sa informacijama o isključenju u tabelu. Ipak, još uvek nećemo izbacivati ništa iz tabele.

In [21]:
# Obeležavanje ROI-eva za isključenje
example_features["exclude_primary_analysis"] = (
    example_features["majority_negative_trace"]
)

example_features["exclusion_reason"] = np.where(
    example_features["majority_negative_trace"],
    "majority_negative_normalized_trace",
    pd.NA,
)

In [22]:
example_features.loc[
    example_features["exclude_primary_analysis"]
]

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_half_width,k_ref_peak_location,k_ref_integral,k_ref_peak_count,k_ref_multiple_responses,k_ref_response_detected,normalized_negative_fraction,majority_negative_trace,exclude_primary_analysis,exclusion_reason
63,ROI_064,BM,1.827039,21.128655,205.0,58.487558,1,False,True,0.586153,True,True,majority_negative_normalized_trace
109,ROI_110,DG,NaN,NaN,NaN,-132.182177,0,False,False,0.998426,True,True,majority_negative_normalized_trace
158,ROI_159,FD,NaN,NaN,NaN,-118.791650,0,False,False,1.000000,True,True,majority_negative_normalized_trace
161,ROI_162,FG,NaN,NaN,NaN,-125.905945,0,False,False,0.998426,True,True,majority_negative_normalized_trace


Vidimo da su 4 ROI-a obeležena prema pravilu, iako jedan od njih pokazuje kalijumski odgovor.

Sada ćemo sačuvati privremenu tabelu u odgovarajući folder, pre nego što pređemo na ekstrakciju parametara fita iz lista 4.

In [23]:
output_path = EXTRACTED_DIR / "example_ALS3_k_features_with_qc.csv"

example_features.to_csv(
    output_path,
    index=False,
)

print(output_path)

/home/pera/Faks/Master/MasterProjekatPetarJ/data/extracted/example_ALS3_k_features_with_qc.csv


#### Ekstrakcija koeficijenata fita

Sada treba da definišemo funkciju koja će izvući parametre jednačine fita iz lista broj 4. Jednačina je sledeća:

\begin{equation}
y = a + b\cdot e^{-\frac{t}{\tau_f}} + c\cdot e^{-\frac{t}{\tau_s}}
\end{equation}

In [24]:
def extract_fit_sheet(
    path: Path,
    sheet_name: str,
    prefix: str,
    expected_n_rois: int,
) -> pd.DataFrame:
    """
    Izvlači parametre fita iz lista gde jedna kolona odgovara jednom parametru.

    Prvi red sadrži oznake parametara.
    Svaki naredni red se poklapa sa po jednim ROI.

    Ukoliko se list završi pre nego što se popune redovi svih ROI-eva,
    nedostajući redovi se popunjavaju NaN vrednostima. 
    """
    sheet = pd.read_excel(
        path,
        sheet_name=sheet_name,
        header=None,
    )

    # Uklanja prvi red (sa oznakama parametara)
    fit_rows = sheet.iloc[1:, :].copy()

    # Konvertuje sve ćelije u numeričke.
    # Prazne ćelije ili one sa greškom postaju NaN
    fit_rows = fit_rows.apply(
        pd.to_numeric,
        errors="coerce",
    )

    # Resetuje indeksiranje redova;
    # Indeks 0 odgovara ROI_001,
    # Indeks 1 odgovara ROI_002, itd.
    fit_rows = fit_rows.reset_index(drop=True)

    # Osigurava da rezultujući DF ima po jedan red za svaki
    # ROI u prethodnom listu.
    # Ako fale redovi u odnosu na prethodni list, ubacuju se NaN redovi
    fit_rows = fit_rows.reindex(range(expected_n_rois))

    output = pd.DataFrame(
        {
            "roi_id": [
                f"ROI_{index:03d}"
                for index in range(1, expected_n_rois + 1)
            ],
            "roi_source_excel_column": [
                excel_column_letter(index)
                for index in range(2, expected_n_rois + 2)
            ],

            f"{prefix}_free_coef": fit_rows.iloc[:, 0].to_numpy(),
            f"{prefix}_free_coef_ci_low": fit_rows.iloc[:, 1].to_numpy(),
            f"{prefix}_free_coef_ci_high": fit_rows.iloc[:, 2].to_numpy(),

            f"{prefix}_coef_fast": fit_rows.iloc[:, 3].to_numpy(),
            f"{prefix}_coef_fast_ci_low": fit_rows.iloc[:, 4].to_numpy(),
            f"{prefix}_coef_fast_ci_high": fit_rows.iloc[:, 5].to_numpy(),

            f"{prefix}_tau_fast": fit_rows.iloc[:, 6].to_numpy(),
            f"{prefix}_tau_fast_ci_low": fit_rows.iloc[:, 7].to_numpy(),
            f"{prefix}_tau_fast_ci_high": fit_rows.iloc[:, 8].to_numpy(),

            f"{prefix}_coef_slow": fit_rows.iloc[:, 9].to_numpy(),
            f"{prefix}_coef_slow_ci_low": fit_rows.iloc[:, 10].to_numpy(),
            f"{prefix}_coef_slow_ci_high": fit_rows.iloc[:, 11].to_numpy(),

            f"{prefix}_tau_slow": fit_rows.iloc[:, 12].to_numpy(),
            f"{prefix}_tau_slow_ci_low": fit_rows.iloc[:, 13].to_numpy(),
            f"{prefix}_tau_slow_ci_high": fit_rows.iloc[:, 14].to_numpy(),

            f"{prefix}_gof": fit_rows.iloc[:, 15].to_numpy(),
            f"{prefix}_percent_fast": fit_rows.iloc[:, 16].to_numpy(),
        }
    )

    # Definisanje kolona koje sadrže centralne parametre (bez CI)
    core_fit_columns = [
        f"{prefix}_free_coef",
        f"{prefix}_coef_fast",
        f"{prefix}_tau_fast",
        f"{prefix}_coef_slow",
        f"{prefix}_tau_slow",
        f"{prefix}_gof",
        f"{prefix}_percent_fast",
    ]

    # Proverava da li su prisutni svi parametri fita
    output[f"{prefix}_fit_available"] = (
        output[core_fit_columns].notna().all(axis=1)
    )

    # Razlikuje između delimično popunjenih i potpuno praznih redova
    output[f"{prefix}_fit_any_value_present"] = (
        output[core_fit_columns].notna().any(axis=1)
    )

    return output

In [25]:
k_fit_features = extract_fit_sheet(
    path=example_path,
    sheet_name="Sheet4",
    prefix="k_ref",
    expected_n_rois=len(k_features),
)

k_fit_features.head()

,roi_id,roi_source_excel_column,k_ref_free_coef,k_ref_free_coef_ci_low,k_ref_free_coef_ci_high,k_ref_coef_fast,k_ref_coef_fast_ci_low,k_ref_coef_fast_ci_high,k_ref_tau_fast,k_ref_tau_fast_ci_low,...,k_ref_coef_slow,k_ref_coef_slow_ci_low,k_ref_coef_slow_ci_high,k_ref_tau_slow,k_ref_tau_slow_ci_low,k_ref_tau_slow_ci_high,k_ref_gof,k_ref_percent_fast,k_ref_fit_available,k_ref_fit_any_value_present
0,ROI_001,B,6.175742e-02,-1.413041,1.536556,4.198518,0.739739,7.657297,28.455556,17.792756,...,0.000018,-2.072803,2.072839,100.000452,-2.957570e+07,2.957590e+07,0.996827,0.999996,True,True
1,ROI_002,C,4.471788e-02,-633.281312,633.370748,2.528793,2.230458,2.827128,16.834815,14.646294,...,0.008394,-633.011047,633.027834,3131.194602,-2.437359e+08,2.437422e+08,0.990993,0.996692,True,True
2,ROI_003,D,1.979960e-09,NaN,NaN,5.943861,5.765319,6.122402,7.050518,6.605311,...,1.050257,0.907977,1.192537,104.398035,7.967756e+01,1.291185e+02,0.987191,0.849837,True,True
3,ROI_004,E,8.900486e-02,-404.518063,404.696073,4.782910,4.656148,4.909673,12.487346,12.002034,...,0.062050,-404.413961,404.538061,3371.914826,-2.256682e+07,2.257356e+07,0.998234,0.987193,True,True
4,ROI_005,F,1.874221e-10,NaN,NaN,4.244487,4.199322,4.289652,11.085828,10.841897,...,0.219589,0.194180,0.244998,100.000000,NaN,NaN,0.997291,0.950810,True,True


In [26]:
k_fit_features.shape

(167, 21)

In [27]:
k_fit_features[
    [
        "roi_id",
        "roi_source_excel_column",
        "k_ref_fit_available",
        "k_ref_fit_any_value_present",
    ]
].tail(10)

,roi_id,roi_source_excel_column,k_ref_fit_available,k_ref_fit_any_value_present
157,ROI_158,FC,True,True
158,ROI_159,FD,False,False
159,ROI_160,FE,True,True
160,ROI_161,FF,True,True
161,ROI_162,FG,False,False
162,ROI_163,FH,True,True
163,ROI_164,FI,True,True
164,ROI_165,FJ,True,True
165,ROI_166,FK,True,True
166,ROI_167,FL,False,False


Vidimo da smo uspešno ekstrahovali sve vrednosti iz lista br. 4 i obeležili redove prema dostupnosti podataka, kao i da je poslednji red popunjen NaN vrednostima.

In [28]:
# Provera broja redova sa dostupnim fitom
k_fit_features["k_ref_fit_available"].value_counts(dropna=False)

k_ref_fit_available
True     163
False      4
Name: count, dtype: int64

In [29]:
# Provera delimično popunjenih redova
k_fit_features.loc[
    k_fit_features["k_ref_fit_any_value_present"]
    & ~k_fit_features["k_ref_fit_available"]
]

,roi_id,roi_source_excel_column,k_ref_free_coef,k_ref_free_coef_ci_low,k_ref_free_coef_ci_high,k_ref_coef_fast,k_ref_coef_fast_ci_low,k_ref_coef_fast_ci_high,k_ref_tau_fast,k_ref_tau_fast_ci_low,...,k_ref_coef_slow,k_ref_coef_slow_ci_low,k_ref_coef_slow_ci_high,k_ref_tau_slow,k_ref_tau_slow_ci_low,k_ref_tau_slow_ci_high,k_ref_gof,k_ref_percent_fast,k_ref_fit_available,k_ref_fit_any_value_present


Vidimo da su svi redovi ili popunjeni ili prazni.

Sada ćemo dodati kolone iz `k_fit_features` tabele u postojeću `example_features` tabelu.

In [30]:
example_features = example_features.merge(
    k_fit_features,
    on=[
        "roi_id",
        "roi_source_excel_column",
    ],
    how="outer",
    validate="one_to_one",
)

example_features.shape

(167, 32)

Dobili smo očekivane dimenzije - imamo po jedan red za svaki ROI, originalnih 13 kolona i 19 novih kolona iz tabele sa parametrima fita.

Sada ćemo proveriti da li se amplitude iz lista 3 poklapaju sa vrednostima parametara iz lista 4. Posmatrajući jednačinu fita, u vremenu $t=0$ (u trenutku vrha kalcijumskog talasa), važi:

\begin{equation}
y(0)=a+b+c
\end{equation}

Kako $a$ predstavlja visinu bazalne linije, maksimalna visina iznad ove linije biće jednaka približno $b+c$.

In [31]:
# Definisanje maksimalne visine
example_features["k_ref_fitted_peak_component"] = (
    example_features["k_ref_coef_fast"]
    + example_features["k_ref_coef_slow"]
)

# Upoređivanje amplitude i maksimalne visine prema fitu
comparison_columns = [
    "k_ref_amplitude",
    "k_ref_fitted_peak_component",
]

valid_comparison = example_features[
    comparison_columns
].dropna()

correlation = valid_comparison.corr().iloc[0, 1]

print(f"Correlation: {correlation:.3f}")

Correlation: 0.988


Na osnovu visoke korelacije možemo tvrditi da su redovi sa vrednostima dobro upareni.

Možemo dodatno proveriti relativnu razliku između izmerene amplitude i $b+c$.

In [32]:
# Definisanje relativne razlike
example_features["k_ref_fit_amplitude_relative_difference"] = (
    (
        example_features["k_ref_amplitude"]
        - example_features["k_ref_fitted_peak_component"]
    ).abs()
    / example_features["k_ref_amplitude"].abs()
)

# Provera relativne razlike
relative_difference = (
    example_features["k_ref_fit_amplitude_relative_difference"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

print(f"Medijalna relativna razlika: {relative_difference.median():.3f}")
print(f"Srednja: {relative_difference.mean():.3f}")
print(f"90-ti postotak: {relative_difference.quantile(0.90):.3f}")
print(f"Maksimum: {relative_difference.max():.3f}")

Medijalna relativna razlika: 0.043
Srednja: 0.050
90-ti postotak: 0.109
Maksimum: 0.163


Vidimo da je poklapanje redova između listova 3 i 4 solidno. Razlika između medijalne i srednje vrednosti razlike verovatno nastaje usled male apsolutne, a velike relativne razlike u ROI-evima gde su detektovane niske amplitude odgovora.

Možemo ispitati redove koji sadrže vrednosti amplitude kalcijumskog odgovora, ali ne i vrednosti parametara fita.

In [33]:
k_response_without_fit = example_features.loc[
    example_features["k_ref_response_detected"]
    & ~example_features["k_ref_fit_available"],
    [
        "roi_id",
        "roi_source_excel_column",
        "k_ref_amplitude",
        "k_ref_half_width",
        "k_ref_peak_location",
        "k_ref_integral",
        "k_ref_fit_available",
    ],
]

k_response_without_fit

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_half_width,k_ref_peak_location,k_ref_integral,k_ref_fit_available
166,ROI_167,FL,6.00528,11.96547,203.0,134.590366,False


Vidimo da tom opisu odgovara ćelija 167. Nju ipak možemo sačuvati za one analize koje se tiču parametara odgovora na kalijum, mada je ne možemo koristiti u analizama koje se oslanjaju na paremetre fita.

Sada ćemo sačuvati novu verziju tabele.

In [34]:
output_path = EXTRACTED_DIR / "example_ALS3_k_features_qc_and_fits.csv"

example_features.to_csv(
    output_path,
    index=False,
)

print(output_path)

/home/pera/Faks/Master/MasterProjekatPetarJ/data/extracted/example_ALS3_k_features_qc_and_fits.csv


### Ekstrakcija podataka o odgovoru na CSF

#### Ekstrakcija parametara kalcijumskog odgovora

Prvo ćemo proveriti kako izgleda list sa parametrima kalcijumskog odgovora na CSF.

In [35]:
csf_summary = pd.read_excel(
    example_path,
    sheet_name="Sheet7",
    header=None,
)

print(csf_summary.shape)

(47, 168)


In [36]:
csf_summary.iloc[:, 0]

0            prvi CSF
1               meanA
2            2.839699
3                 NaN
4                 NaN
5                 NaN
6                 NaN
7                 NaN
8                 NaN
9                 NaN
10                NaN
11                NaN
12     amplitude CSF1
13                NaN
14                NaN
15                NaN
16                NaN
17                NaN
18                NaN
19                NaN
20                NaN
21                NaN
22                NaN
23    polusirine CSF1
24                NaN
25                NaN
26                NaN
27                NaN
28                NaN
29                NaN
30                NaN
31                NaN
32                NaN
33                NaN
34      lokacije CSF1
35                NaN
36                NaN
37                NaN
38                NaN
39                NaN
40                NaN
41                NaN
42                NaN
43                NaN
44                NaN
45      in

Budući da je struktura listova koji sadrže podatke u vezi sa tretmanom ćelija CSF-om identična onoj u listovima sa podacima o odgovoru na K<sup>+</sup>, možemo upotrebiti već napisane funkcije, vodeći računa o listovima iz kojih izvlačimo vrednosti.

In [37]:
# Ekstrakcija vrednosti iz lista 7 i provera tabele
csf_features = extract_summary_sheet(
    path=example_path,
    sheet_name="Sheet7",
    prefix="csf",
)

csf_features.head()

,roi_id,roi_source_excel_column,csf_amplitude,csf_half_width,csf_peak_location,csf_integral,csf_peak_count,csf_multiple_responses,csf_response_detected
0,ROI_001,B,2.476320,18.055572,404.0,53.499582,1,False,True
1,ROI_002,C,1.034618,13.775792,404.0,19.353031,1,False,True
2,ROI_003,D,3.479419,10.227863,403.0,51.276839,1,False,True
3,ROI_004,E,2.435517,14.078715,403.0,43.434964,1,False,True
4,ROI_005,F,2.421803,10.084101,403.0,34.923052,1,False,True


In [38]:
csf_features.shape

(167, 9)

Dimenzije su odhgovarajuće. Sada ćemo proveriti brojčano stanje ćelija koje su odgovorile na tretman. 

In [39]:
csf_features["csf_response_detected"].value_counts(dropna=False)

csf_response_detected
True     147
False     20
Name: count, dtype: int64

In [40]:
csf_features["csf_multiple_responses"].value_counts(dropna=False)

csf_multiple_responses
False    167
Name: count, dtype: int64

In [41]:
csf_features["csf_peak_count"].value_counts(dropna=False).sort_index()

csf_peak_count
0     20
1    147
Name: count, dtype: int64

Vidimo da je zabeleženo 147 ROI-eva u kojima je primećen odgovor, i da su svi odgovori sa po jednim pikom.

Sada možemo preći na ekstrakciju parametara fita iz lista 8.

#### Ekstrakcija koeficijenata fita

In [42]:
# Ekstrakcija i provera
csf_fit_features = extract_fit_sheet(
    path=example_path,
    sheet_name="Sheet8",
    prefix="csf",
    expected_n_rois=len(csf_features),
)

csf_fit_features.head()

,roi_id,roi_source_excel_column,csf_free_coef,csf_free_coef_ci_low,csf_free_coef_ci_high,csf_coef_fast,csf_coef_fast_ci_low,csf_coef_fast_ci_high,csf_tau_fast,csf_tau_fast_ci_low,...,csf_coef_slow,csf_coef_slow_ci_low,csf_coef_slow_ci_high,csf_tau_slow,csf_tau_slow_ci_low,csf_tau_slow_ci_high,csf_gof,csf_percent_fast,csf_fit_available,csf_fit_any_value_present
0,ROI_001,B,7.090837e-08,-0.014271,0.014271,2.830691,2.782123,2.879259,16.634459,16.137859,...,1.253591e-09,NaN,NaN,100.000001,-1.671599e+09,1.671599e+09,0.987547,1.000000,True,True
1,ROI_002,C,7.619440e-03,-0.033288,0.048527,1.156284,1.115646,1.196922,12.570237,11.885760,...,1.262480e-03,-0.021703,0.024228,110.716331,-1.026109e+04,1.048252e+04,0.991998,0.998909,True,True
2,ROI_003,D,7.602005e-03,-1.033136,1.048340,3.837540,3.760105,3.914974,9.076752,8.693729,...,3.833144e-02,-0.947216,1.023879,351.842150,-1.296392e+04,1.366761e+04,0.989249,0.990110,True,True
3,ROI_004,E,1.204879e-06,-0.027179,0.027182,2.791426,2.740419,2.842432,13.507179,13.088005,...,6.523812e-09,NaN,NaN,100.000106,-1.103682e+09,1.103682e+09,0.989059,1.000000,True,True
4,ROI_005,F,2.010633e-02,-0.020935,0.061148,2.502818,2.467348,2.538287,10.370744,10.107288,...,2.032103e-04,-0.022898,0.023304,110.942701,-6.104711e+04,6.126899e+04,0.997567,0.999919,True,True


In [43]:
csf_fit_features.shape

(167, 21)

In [44]:
csf_fit_features["csf_fit_available"].value_counts(dropna=False)

csf_fit_available
True     148
False     19
Name: count, dtype: int64

Dimenzije su OK, i vidimo da su vrednosti fita dostupne za 148 ROI-a - za jedan više nego što ima zabeleženih odgovora. Treba proveriti koji ROI ima vrednost fita ali ne i parametre odgovora. Prvo ćemo pogledati da li ima nekih redova gde su parametri fita delimično popunjeni.

In [45]:
csf_fit_features.loc[
    csf_fit_features["csf_fit_any_value_present"]
    & ~csf_fit_features["csf_fit_available"]
]

,roi_id,roi_source_excel_column,csf_free_coef,csf_free_coef_ci_low,csf_free_coef_ci_high,csf_coef_fast,csf_coef_fast_ci_low,csf_coef_fast_ci_high,csf_tau_fast,csf_tau_fast_ci_low,...,csf_coef_slow,csf_coef_slow_ci_low,csf_coef_slow_ci_high,csf_tau_slow,csf_tau_slow_ci_low,csf_tau_slow_ci_high,csf_gof,csf_percent_fast,csf_fit_available,csf_fit_any_value_present


Kao i malopre, vrednosti fita su ili sve prisutne ili potpuno odsutne. Sada ćemo spojiti tabele sa parametrima odgovora i fita.

In [46]:
csf_combined = csf_features.merge(
    csf_fit_features,
    on=[
        "roi_id",
        "roi_source_excel_column",
    ],
    how="outer",
    validate="one_to_one",
)

csf_combined.shape

(167, 28)

Ova tabela ima 4 kolone manje nego što je to bio slučaj sa odgovorom na kalijum, što je očekivano (u toj tabeli smo beležili koji ROI-evi će potencijalno biti isključeni).

Sada kad smo spojili sve parametre tretmana CSF-om, možemo proveriti koja ćelija ima vrednosti fita, ali ne i detektovan odgovor.

In [47]:
csf_combined.loc[
    csf_combined["csf_fit_available"]
    & ~csf_combined["csf_response_detected"]
]

,roi_id,roi_source_excel_column,csf_amplitude,csf_half_width,csf_peak_location,csf_integral,csf_peak_count,csf_multiple_responses,csf_response_detected,csf_free_coef,...,csf_coef_slow,csf_coef_slow_ci_low,csf_coef_slow_ci_high,csf_tau_slow,csf_tau_slow_ci_low,csf_tau_slow_ci_high,csf_gof,csf_percent_fast,csf_fit_available,csf_fit_any_value_present
165,ROI_166,FK,NaN,NaN,NaN,0.0,0,False,False,2.306583e-11,...,0.413214,0.375604,0.450824,100.0,NaN,NaN,0.9972,0.932203,True,True


Ovaj slučaj verovatno je posledica dupliranih vrednosti parametara fita koje se nalaze pri dnu listova u fajlovima sa eksperimentalnim podacima. Ova problematika biće razmatrana u delu **Ispitivanje dupliranih vrednosti**.

Sada ćemo spojiti tabelu sa parametrima odgovora na CSF sa postojećom K<sup>+</sup> tabelom.

In [48]:
example_features = example_features.merge(
    csf_combined,
    on=[
        "roi_id",
        "roi_source_excel_column",
    ],
    how="outer",
    validate="one_to_one",
)

example_features.shape

(167, 60)

Dimenzije izgledaju OK. Sada ćemo proveriti da li su ROI-evi u listovima 7 i 8 dobro upareni.

In [49]:
example_features["csf_fitted_peak_component"] = (
    example_features["csf_coef_fast"]
    + example_features["csf_coef_slow"]
)

In [50]:
csf_comparison_columns = [
    "csf_amplitude",
    "csf_fitted_peak_component",
]

valid_csf_comparison = example_features[
    csf_comparison_columns
].dropna()

csf_correlation = valid_csf_comparison.corr().iloc[0, 1]

print(f"Korelacija između redova: {csf_correlation:.3f}")

Korelacija između redova: 0.995


Vrednost je visoka i ukazuje na to da su redovi dobro upareni. Kao i malopre, izračunaćemo i relativnu razliku u intenzitetima pikova i izračunatim $b+c$ vrednostima.

In [51]:
example_features["csf_fit_amplitude_relative_difference"] = (
    (
        example_features["csf_amplitude"]
        - example_features["csf_fitted_peak_component"]
    ).abs()
    / example_features["csf_amplitude"].abs()
)

In [52]:
csf_relative_difference = (
    example_features["csf_fit_amplitude_relative_difference"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

print(f"Medijalna relativna razlika: {csf_relative_difference.median():.3f}")
print(f"Srednja: {csf_relative_difference.mean():.3f}")
print(f"90-ti postotak: {csf_relative_difference.quantile(0.90):.3f}")
print(f"Maksimum: {csf_relative_difference.max():.3f}")

Medijalna relativna razlika: 0.103
Srednja: 0.098
90-ti postotak: 0.144
Maksimum: 0.209


Rezultati su OK. Sada ćemo proveriti da li ima ćelija koje su pokazale odgovore, ali koji nisu fitovani.

In [53]:
csf_response_without_fit = example_features.loc[
    example_features["csf_response_detected"]
    & ~example_features["csf_fit_available"],
    [
        "roi_id",
        "roi_source_excel_column",
        "csf_amplitude",
        "csf_half_width",
        "csf_peak_location",
        "csf_integral",
        "csf_fit_available",
    ],
]

csf_response_without_fit

,roi_id,roi_source_excel_column,csf_amplitude,csf_half_width,csf_peak_location,csf_integral,csf_fit_available


Ovaj put, takvih ćelija nema. Sada ćemo proveriti da li postoje vijabilne ćelije (koje su pokazale odgovor na K<sup>+</sup>) koje nisu odgovorile na tretman CSF-om.

In [54]:
k_responders_without_csf_peak = example_features.loc[
    example_features["k_ref_response_detected"]
    & ~example_features["csf_response_detected"]
    & ~example_features["majority_negative_trace"],
    [
        "roi_id",
        "roi_source_excel_column",
        "k_ref_amplitude",
        "k_ref_integral",
        "csf_integral",
        "csf_response_detected",
    ],
]

k_responders_without_csf_peak.head()

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_integral,csf_integral,csf_response_detected
6,ROI_007,H,3.734535,83.071027,10.562382,False
18,ROI_019,T,3.110535,95.933659,20.002583,False
50,ROI_051,AZ,3.416666,99.540312,39.460299,False
51,ROI_052,BA,3.007430,92.260629,6.903528,False
52,ROI_053,BB,2.592455,74.781752,6.776486,False


In [55]:
print("Broj vijabilnih ćelija bez odgovora na CSF:", 
      len(k_responders_without_csf_peak))

Broj vijabilnih ćelija bez odgovora na CSF: 16


Sada ćemo pogledati kolone završene, kombinovane tabele.

In [56]:
example_features.columns.tolist()

['roi_id',
 'roi_source_excel_column',
 'k_ref_amplitude',
 'k_ref_half_width',
 'k_ref_peak_location',
 'k_ref_integral',
 'k_ref_peak_count',
 'k_ref_multiple_responses',
 'k_ref_response_detected',
 'normalized_negative_fraction',
 'majority_negative_trace',
 'exclude_primary_analysis',
 'exclusion_reason',
 'k_ref_free_coef',
 'k_ref_free_coef_ci_low',
 'k_ref_free_coef_ci_high',
 'k_ref_coef_fast',
 'k_ref_coef_fast_ci_low',
 'k_ref_coef_fast_ci_high',
 'k_ref_tau_fast',
 'k_ref_tau_fast_ci_low',
 'k_ref_tau_fast_ci_high',
 'k_ref_coef_slow',
 'k_ref_coef_slow_ci_low',
 'k_ref_coef_slow_ci_high',
 'k_ref_tau_slow',
 'k_ref_tau_slow_ci_low',
 'k_ref_tau_slow_ci_high',
 'k_ref_gof',
 'k_ref_percent_fast',
 'k_ref_fit_available',
 'k_ref_fit_any_value_present',
 'k_ref_fitted_peak_component',
 'k_ref_fit_amplitude_relative_difference',
 'csf_amplitude',
 'csf_half_width',
 'csf_peak_location',
 'csf_integral',
 'csf_peak_count',
 'csf_multiple_responses',
 'csf_response_detected',
 '

In [57]:
# Grupisanje kolona prema prefiksu
for prefix in [
    "k_ref",
    "csf",
]:
    print(f"\n--- {prefix} kolone ---")

    for column in example_features.columns:
        if column.startswith(prefix):
            print(column)


--- k_ref kolone ---
k_ref_amplitude
k_ref_half_width
k_ref_peak_location
k_ref_integral
k_ref_peak_count
k_ref_multiple_responses
k_ref_response_detected
k_ref_free_coef
k_ref_free_coef_ci_low
k_ref_free_coef_ci_high
k_ref_coef_fast
k_ref_coef_fast_ci_low
k_ref_coef_fast_ci_high
k_ref_tau_fast
k_ref_tau_fast_ci_low
k_ref_tau_fast_ci_high
k_ref_coef_slow
k_ref_coef_slow_ci_low
k_ref_coef_slow_ci_high
k_ref_tau_slow
k_ref_tau_slow_ci_low
k_ref_tau_slow_ci_high
k_ref_gof
k_ref_percent_fast
k_ref_fit_available
k_ref_fit_any_value_present
k_ref_fitted_peak_component
k_ref_fit_amplitude_relative_difference

--- csf kolone ---
csf_amplitude
csf_half_width
csf_peak_location
csf_integral
csf_peak_count
csf_multiple_responses
csf_response_detected
csf_free_coef
csf_free_coef_ci_low
csf_free_coef_ci_high
csf_coef_fast
csf_coef_fast_ci_low
csf_coef_fast_ci_high
csf_tau_fast
csf_tau_fast_ci_low
csf_tau_fast_ci_high
csf_coef_slow
csf_coef_slow_ci_low
csf_coef_slow_ci_high
csf_tau_slow
csf_tau_slow

Sada ćemo sačuvati kompletnu tabelu kao .csv fajl.

In [58]:
output_path = (
    EXTRACTED_DIR
    / "example_ALS3_k_and_csf_features_with_qc.csv"
)

example_features.to_csv(
    output_path,
    index=False,
)

print(output_path)

/home/pera/Faks/Master/MasterProjekatPetarJ/data/extracted/example_ALS3_k_and_csf_features_with_qc.csv


#### Ispitivanje dupliranih vrednosti

Videli smo da ROI_166 ima vrednosti fita, iako nije zabeležen odgovor na tretman CSF-om. Ovo ne bi trebalo da se dešava, budući da se parametri fita dobijaju direktno iz parametara odgovora. 

In [59]:
# Pregled ROI_166
example_features.loc[
    example_features["roi_id"] == "ROI_166",
    [
        "roi_id",
        "roi_source_excel_column",
        "majority_negative_trace",

        "k_ref_amplitude",
        "k_ref_integral",
        "k_ref_fit_available",

        "csf_amplitude",
        "csf_half_width",
        "csf_peak_location",
        "csf_integral",
        "csf_response_detected",

        "csf_free_coef",
        "csf_coef_fast",
        "csf_tau_fast",
        "csf_coef_slow",
        "csf_tau_slow",
        "csf_gof",
        "csf_percent_fast",
        "csf_fit_available",
    ],
]

,roi_id,roi_source_excel_column,majority_negative_trace,k_ref_amplitude,k_ref_integral,k_ref_fit_available,csf_amplitude,csf_half_width,csf_peak_location,csf_integral,csf_response_detected,csf_free_coef,csf_coef_fast,csf_tau_fast,csf_coef_slow,csf_tau_slow,csf_gof,csf_percent_fast,csf_fit_available
165,ROI_166,FK,False,5.886173,115.736056,True,NaN,NaN,NaN,0.0,False,2.306583e-11,5.681694,11.893803,0.413214,100.0,0.9972,0.932203,True


Prilikom pregleda tabele sa eksperimentalnim podacima, primećeno je da su neki od poslednjih redova u listovima koji sadrže vrednosti parametara fita u različitim tretmanima identični. To može ukazivati na grešku u popunjavanju tabela, pa ćemo ispitati ove redove.

In [60]:
for sheet_name in ["Sheet4", "Sheet6", "Sheet8"]:
    sheet = pd.read_excel(
        example_path,
        sheet_name=sheet_name,
        header=None,
    )

    print(f"\n--- {sheet_name}: Poslednji redovi ---")
    display(sheet.iloc[-3:, :])


--- Sheet4: Poslednji redovi ---


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
164,0.0,NaN,NaN,4.43199,4.289178,4.574802,18.727465,17.386082,20.068848,0.895274,0.7637,1.026848,100.0,NaN,NaN,0.984056,0.831945
165,0.03741,-1.961311,2.03613,5.079292,1.445328,8.713255,25.146759,15.90698,34.386538,0.164178,-1.62653,1.954886,100.000492,-3896.310968,4096.311952,0.99494,0.968689
166,0.0,NaN,NaN,5.681694,5.619188,5.7442,11.893803,11.618852,12.168753,0.413214,0.375604,0.450824,100.0,NaN,NaN,0.9972,0.932203



--- Sheet6: Poslednji redovi ---


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
164,0.366101,0.346068,0.386135,4.128866,4.081096,4.176637,20.913572,20.410521,21.416624,0.0,NaN,NaN,100.000001,-1031026592.128734,1031026792.128735,0.994902,1.0
165,0.000001,-0.087112,0.087114,3.976469,3.835853,4.117085,17.818471,16.732208,18.904733,0.734703,0.499111,0.970295,100.0,NaN,NaN,0.993576,0.844051
166,0.0,NaN,NaN,5.681694,5.619188,5.7442,11.893803,11.618852,12.168753,0.413214,0.375604,0.450824,100.0,NaN,NaN,0.9972,0.932203



--- Sheet8: Poslednji redovi ---


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
164,0.052339,-19.349498,19.454175,4.48202,4.395963,4.568076,12.69861,12.204458,13.192763,0.008289,-19.316628,19.333206,1372.45155,-3526165.73421,3528910.637311,0.993084,0.998154
165,0.000001,-0.087112,0.087114,3.976469,3.835853,4.117085,17.818471,16.732208,18.904733,0.734703,0.499111,0.970295,100.0,NaN,NaN,0.993576,0.844051
166,0.0,NaN,NaN,5.681694,5.619188,5.7442,11.893803,11.618852,12.168753,0.413214,0.375604,0.450824,100.0,NaN,NaN,0.9972,0.932203


Vidimo da je ROI_166 identičan na sva tri lista, dok se vrednost ROI_165 poklapaju u listovima 6 i 8.

Budući da ova pojava ukazuje na mogući problem prilikom beleženja vrednosti parametar fita u odgovarajuće redove, napisaćemo funkciju koja automatski detektuje duplirane vrednosti.

In [61]:
def detect_cross_sheet_fit_duplicates(
    path: Path,
    fit_sheet_names: list[str],
) -> pd.DataFrame:
    """
    Detektuje redove sa parametrima fita koji nisu prazni, a sadrže 
    identične vrednosti u više od jednog lista.

    Vraća jedan red za svaki slučaj ponavljanja.
    """

    # Kolone sa 7 centralnih parametara fita
    core_fit_column_indices = [
        0,
        3,
        6,
        9,
        12,
        15,
        16,
    ]

    all_fit_rows = []

    for sheet_name in fit_sheet_names:
        sheet = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=None,
        )

        # Preskače prvi red i konvertuje sve ćelije u numerički oblik
        fit_rows = (
            sheet.iloc[1:, :]
            .apply(pd.to_numeric, errors="coerce")
            .reset_index(drop=True)
        )

        for row_index, row in fit_rows.iterrows():
            roi_number = row_index + 1

            core_values = row.iloc[
                core_fit_column_indices
            ]

            # Poredi sve redove sa centralnim parametrima fita
            if core_values.notna().all():
                all_fit_rows.append(
                    {
                        "fit_sheet": sheet_name,
                        "roi_id": f"ROI_{roi_number:03d}",
                        "source_fit_excel_row": roi_number + 1,
                        "fit_signature": tuple(
                            core_values.to_numpy()
                        ),
                    }
                )

    duplicate_table = pd.DataFrame(all_fit_rows)

    if duplicate_table.empty:
        return duplicate_table

    # Broji koliko listova sadrži svaki dupliran red
    sheet_counts = (
        duplicate_table
        .groupby("fit_signature")["fit_sheet"]
        .transform("nunique")
    )

    duplicate_table["fit_duplicate_across_sheets"] = (
        sheet_counts > 1
    )

    # Kreira tabelu sa informacijama o duplikatima
    occurrence_labels = (
        duplicate_table
        .assign(
            occurrence=lambda df: (
                df["fit_sheet"]
                + ":row_"
                + df["source_fit_excel_row"].astype(str)
            )
        )
        .groupby("fit_signature")["occurrence"]
        .transform(
            lambda values: ", ".join(sorted(values))
        )
    )

    duplicate_table["matching_occurrences"] = (
        occurrence_labels
    )

    return duplicate_table

In [62]:
# Izvršenje funkcije
example_fit_duplicates = (
    detect_cross_sheet_fit_duplicates(
        path=example_path,
        fit_sheet_names=[
            "Sheet4",
            "Sheet6",
            "Sheet8",
        ],
    )
)

suspicious_fit_rows = example_fit_duplicates.loc[
    example_fit_duplicates[
        "fit_duplicate_across_sheets"
    ]
].copy()

In [63]:
# Prikaz zabeleženih redova
suspicious_fit_rows[
    [
        "fit_sheet",
        "roi_id",
        "source_fit_excel_row",
        "matching_occurrences",
    ]
]

,fit_sheet,roi_id,source_fit_excel_row,matching_occurrences
162,Sheet4,ROI_166,167,"Sheet4:row_167, Sheet6:row_167, Sheet8:row_167"
324,Sheet6,ROI_165,166,"Sheet6:row_166, Sheet8:row_166"
325,Sheet6,ROI_166,167,"Sheet4:row_167, Sheet6:row_167, Sheet8:row_167"
472,Sheet8,ROI_165,166,"Sheet6:row_166, Sheet8:row_166"
473,Sheet8,ROI_166,167,"Sheet4:row_167, Sheet6:row_167, Sheet8:row_167"


In [64]:
suspicious_fit_rows["fit_sheet"].value_counts()

fit_sheet
Sheet6    2
Sheet8    2
Sheet4    1
Name: count, dtype: int64

Sada vidimo koji redovi su duplirani i gde se nalaze.

Sledeći korak je beleženje redova čije su vrednosti duplirane u listovima specifično za parametre fita odgovora na K<sup>+</sup> i/ili na CSF, zato što dati ROI može imati problematičan fit za K<sup>+</sup>, CSF, oba, ili nijedan.

In [65]:
# Kalijum-specifični duplikati
k_duplicate_flags = (
    example_fit_duplicates.loc[
        example_fit_duplicates["fit_sheet"] == "Sheet4",
        [
            "roi_id",
            "fit_duplicate_across_sheets",
            "matching_occurrences",
        ],
    ]
    .rename(
        columns={
            "fit_duplicate_across_sheets": "k_ref_fit_duplicate_flag",
            "matching_occurrences": "k_ref_fit_duplicate_matches",
        }
    )
)

In [66]:
# CSF-specifični duplikati
csf_duplicate_flags = (
    example_fit_duplicates.loc[
        example_fit_duplicates["fit_sheet"] == "Sheet8",
        [
            "roi_id",
            "fit_duplicate_across_sheets",
            "matching_occurrences",
        ],
    ]
    .rename(
        columns={
            "fit_duplicate_across_sheets": "csf_fit_duplicate_flag",
            "matching_occurrences": "csf_fit_duplicate_matches",
        }
    )
)

In [67]:
# Spajanje sa čitavom tabelom
example_features = (
    example_features
    .merge(
        k_duplicate_flags,
        on="roi_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        csf_duplicate_flags,
        on="roi_id",
        how="left",
        validate="one_to_one",
    )
)

Na osnovu dosadašnjeg koda, ROI-evi sa nekompletnim parametrima fita neće biti obeleženi, pa ćemo ih označiti sa `False`.

In [68]:
for column in [
    "k_ref_fit_duplicate_flag",
    "csf_fit_duplicate_flag",
]:
    example_features[column] = (
        example_features[column]
        .fillna(False)
        .astype(bool)
    )

In [69]:
# Obeležavanje redova sa validnim fitovima
example_features["k_ref_fit_valid"] = (
    example_features["k_ref_fit_available"]
    & ~example_features["k_ref_fit_duplicate_flag"]
)

example_features["csf_fit_valid"] = (
    example_features["csf_fit_available"]
    & ~example_features["csf_fit_duplicate_flag"]
)

Na kraju ćemo pogledati redove koji su obeleženi kao duplirani.

In [70]:
flagged_fit_rows = example_features.loc[
    (
        example_features["k_ref_fit_duplicate_flag"]
        | example_features["csf_fit_duplicate_flag"]
    ),
    [
        "roi_id",
        "roi_source_excel_column",

        "k_ref_amplitude",
        "k_ref_fit_available",
        "k_ref_fit_duplicate_flag",
        "k_ref_fit_valid",
        "k_ref_fit_duplicate_matches",

        "csf_amplitude",
        "csf_response_detected",
        "csf_fit_available",
        "csf_fit_duplicate_flag",
        "csf_fit_valid",
        "csf_fit_duplicate_matches",
    ],
]

flagged_fit_rows

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_fit_available,k_ref_fit_duplicate_flag,k_ref_fit_valid,k_ref_fit_duplicate_matches,csf_amplitude,csf_response_detected,csf_fit_available,csf_fit_duplicate_flag,csf_fit_valid,csf_fit_duplicate_matches
164,ROI_165,FJ,5.106248,True,False,True,Sheet4:row_166,4.26409,True,True,True,False,"Sheet6:row_166, Sheet8:row_166"
165,ROI_166,FK,5.886173,True,True,False,"Sheet4:row_167, Sheet6:row_167, Sheet8:row_167",NaN,False,True,True,False,"Sheet4:row_167, Sheet6:row_167, Sheet8:row_167"


Vidimo relevantne informacije za dva problematična reda. ROI_165 ima dupliran CSF fit, a ROI_166 duplirana oba fita.

#### Poslednje provere

Pre nego što pređemo na generalizaciju dosadašnjih funkcija, predstavićemo kratak pregled nedostajućih i/ili problematičnih vrednosti sa kojima smo se uglavnom već susreli.

In [71]:
# Prisutan CSF fit bez detektovanog pika
example_features.loc[
    example_features["csf_fit_available"]
    & ~example_features["csf_response_detected"],
    [
        "roi_id",
        "roi_source_excel_column",
        "csf_amplitude",
        "csf_integral",
        "csf_fit_available",
        "csf_fit_duplicate_flag",
        "csf_fit_valid",
    ],
]

,roi_id,roi_source_excel_column,csf_amplitude,csf_integral,csf_fit_available,csf_fit_duplicate_flag,csf_fit_valid
165,ROI_166,FK,NaN,0.0,True,True,False


In [72]:
# Prisutan odgovor na kalijum ali bez validnog fita
example_features.loc[
    example_features["k_ref_response_detected"]
    & ~example_features["k_ref_fit_valid"],
    [
        "roi_id",
        "roi_source_excel_column",
        "k_ref_amplitude",
        "k_ref_integral",
        "k_ref_fit_available",
        "k_ref_fit_duplicate_flag",
        "k_ref_fit_valid",
    ],
]

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_integral,k_ref_fit_available,k_ref_fit_duplicate_flag,k_ref_fit_valid
165,ROI_166,FK,5.886173,115.736056,True,True,False
166,ROI_167,FL,6.005280,134.590366,False,False,False


Vidimo da imamo dva problematična ROI-a - ROI_166 (dupliran fit) i ROI_167 (nedostajući fit).

In [73]:
# Validan odgovor na kalijum ali bez detektovanog CSF pika
k_responders_without_csf_peak = example_features.loc[
    example_features["k_ref_response_detected"]
    & ~example_features["csf_response_detected"]
    & ~example_features["majority_negative_trace"],
    [
        "roi_id",
        "roi_source_excel_column",
        "k_ref_amplitude",
        "k_ref_integral",
        "csf_integral",
        "csf_response_detected",
    ],
]

print("Broj vijabilnih ćelija bez odgovora na CSF:", 
      len(k_responders_without_csf_peak))

Broj vijabilnih ćelija bez odgovora na CSF: 16


In [74]:
# Redovi obeleženi za isključenje iz analize
# na osnovu pretežno negativnih normalizovanih
# vrednosti fluorescencije
example_features.loc[
    example_features["exclude_primary_analysis"]
]

,roi_id,roi_source_excel_column,k_ref_amplitude,k_ref_half_width,k_ref_peak_location,k_ref_integral,k_ref_peak_count,k_ref_multiple_responses,k_ref_response_detected,normalized_negative_fraction,...,csf_fit_available,csf_fit_any_value_present,csf_fitted_peak_component,csf_fit_amplitude_relative_difference,k_ref_fit_duplicate_flag,k_ref_fit_duplicate_matches,csf_fit_duplicate_flag,csf_fit_duplicate_matches,k_ref_fit_valid,csf_fit_valid
63,ROI_064,BM,1.827039,21.128655,205.0,58.487558,1,False,True,0.586153,...,False,False,NaN,NaN,False,Sheet4:row_65,False,NaN,True,False
109,ROI_110,DG,NaN,NaN,NaN,-132.182177,0,False,False,0.998426,...,False,False,NaN,NaN,False,NaN,False,NaN,False,False
158,ROI_159,FD,NaN,NaN,NaN,-118.791650,0,False,False,1.000000,...,False,False,NaN,NaN,False,NaN,False,NaN,False,False
161,ROI_162,FG,NaN,NaN,NaN,-125.905945,0,False,False,0.998426,...,False,False,NaN,NaN,False,NaN,False,NaN,False,False


Pogledaćemo kako izgleda poslednja verzija tabele.

In [75]:
example_features.columns.tolist()

['roi_id',
 'roi_source_excel_column',
 'k_ref_amplitude',
 'k_ref_half_width',
 'k_ref_peak_location',
 'k_ref_integral',
 'k_ref_peak_count',
 'k_ref_multiple_responses',
 'k_ref_response_detected',
 'normalized_negative_fraction',
 'majority_negative_trace',
 'exclude_primary_analysis',
 'exclusion_reason',
 'k_ref_free_coef',
 'k_ref_free_coef_ci_low',
 'k_ref_free_coef_ci_high',
 'k_ref_coef_fast',
 'k_ref_coef_fast_ci_low',
 'k_ref_coef_fast_ci_high',
 'k_ref_tau_fast',
 'k_ref_tau_fast_ci_low',
 'k_ref_tau_fast_ci_high',
 'k_ref_coef_slow',
 'k_ref_coef_slow_ci_low',
 'k_ref_coef_slow_ci_high',
 'k_ref_tau_slow',
 'k_ref_tau_slow_ci_low',
 'k_ref_tau_slow_ci_high',
 'k_ref_gof',
 'k_ref_percent_fast',
 'k_ref_fit_available',
 'k_ref_fit_any_value_present',
 'k_ref_fitted_peak_component',
 'k_ref_fit_amplitude_relative_difference',
 'csf_amplitude',
 'csf_half_width',
 'csf_peak_location',
 'csf_integral',
 'csf_peak_count',
 'csf_multiple_responses',
 'csf_response_detected',
 '

In [76]:
# Grupisanje kolona prema prefiksu
for prefix in [
    "k_ref",
    "csf",
]:
    print(f"\n--- {prefix} kolone ---")

    for column in example_features.columns:
        if column.startswith(prefix):
            print(column)


--- k_ref kolone ---
k_ref_amplitude
k_ref_half_width
k_ref_peak_location
k_ref_integral
k_ref_peak_count
k_ref_multiple_responses
k_ref_response_detected
k_ref_free_coef
k_ref_free_coef_ci_low
k_ref_free_coef_ci_high
k_ref_coef_fast
k_ref_coef_fast_ci_low
k_ref_coef_fast_ci_high
k_ref_tau_fast
k_ref_tau_fast_ci_low
k_ref_tau_fast_ci_high
k_ref_coef_slow
k_ref_coef_slow_ci_low
k_ref_coef_slow_ci_high
k_ref_tau_slow
k_ref_tau_slow_ci_low
k_ref_tau_slow_ci_high
k_ref_gof
k_ref_percent_fast
k_ref_fit_available
k_ref_fit_any_value_present
k_ref_fitted_peak_component
k_ref_fit_amplitude_relative_difference
k_ref_fit_duplicate_flag
k_ref_fit_duplicate_matches
k_ref_fit_valid

--- csf kolone ---
csf_amplitude
csf_half_width
csf_peak_location
csf_integral
csf_peak_count
csf_multiple_responses
csf_response_detected
csf_free_coef
csf_free_coef_ci_low
csf_free_coef_ci_high
csf_coef_fast
csf_coef_fast_ci_low
csf_coef_fast_ci_high
csf_tau_fast
csf_tau_fast_ci_low
csf_tau_fast_ci_high
csf_coef_slow